# DriveSafe AI — Train and Evaluate an Eye-State CNN

This notebook trains a binary CNN on **your labeled eye image dataset** and evaluates it on a separate test folder.

## Dataset format required
Upload a ZIP file with this exact folder structure (folder names are case-sensitive):

```
train/
  open/
  closed/
test/
  open/
  closed/
```

Use a legitimate public dataset and follow its license/terms. If possible, split by person/subject so the same person's images do not appear in both train and test. This makes evaluation more trustworthy.

The model expects eye crops resized to 64×64 RGB images. The live backend uses MediaPipe to crop eye regions, so performance depends on whether the webcam crops resemble the dataset images. This is a prototype, not a certified safety device.


In [ ]:
# Install packages
!pip -q install scikit-learn seaborn


In [ ]:
# Upload a ZIP dataset from your computer.
from google.colab import files
uploaded = files.upload()  # choose eye_dataset.zip


In [ ]:
import zipfile, os, pathlib

zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError("Please upload a .zip file containing train/open, train/closed, test/open and test/closed folders.")

DATA_DIR = "/content/eye_dataset"
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATA_DIR)

# Find the folder that contains train/ and test/, even if the ZIP has one extra wrapper folder.
def find_dataset_root(path):
    path = pathlib.Path(path)
    for candidate in [path] + list(path.rglob("*")):
        if candidate.is_dir() and (candidate/"train"/"open").is_dir() and (candidate/"train"/"closed").is_dir() and (candidate/"test"/"open").is_dir() and (candidate/"test"/"closed").is_dir():
            return candidate
    return None

ROOT = find_dataset_root(DATA_DIR)
if ROOT is None:
    raise ValueError("Expected train/open, train/closed, test/open and test/closed. Check the ZIP folder structure.")
print("Dataset root:", ROOT)
for split in ["train", "test"]:
    for label in ["open", "closed"]:
        folder = ROOT/split/label
        imgs = [p for p in folder.iterdir() if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}]
        print(f"{split}/{label}: {len(imgs)} images")


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score
from pathlib import Path

SEED = 42
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
tf.keras.utils.set_random_seed(SEED)

train_dir = str(ROOT / "train")
test_dir = str(ROOT / "test")

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, labels="inferred", label_mode="binary",
    class_names=["closed", "open"], image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, labels="inferred", label_mode="binary",
    class_names=["closed", "open"], image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=False
)
class_names = train_ds.class_names
print("Label mapping:", {name:i for i,name in enumerate(class_names)})
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


In [ ]:
# Inspect a few training examples to verify labels are correct.
plt.figure(figsize=(10, 5))
for images, labels in train_ds.take(1):
    for i in range(min(10, len(images))):
        ax = plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_index = int(labels[i].numpy()[0])
        plt.title(class_names[label_index])
        plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Build a compact CNN. Images are rescaled to 0..1 inside the model.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.08),
], name="augmentation")

model = keras.Sequential([
    layers.Input(shape=(64, 64, 3)),
    data_augmentation,
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(1, activation="sigmoid", name="open_probability")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")]
)
model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("eye_cnn.keras", monitor="val_loss", save_best_only=True)
]

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    callbacks=callbacks
)


In [ ]:
# Plot training curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Test accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.title("Accuracy")
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Test loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("Loss")
plt.tight_layout(); plt.show()


In [ ]:
# Evaluate on held-out test images and print report metrics.
y_true = np.concatenate([labels.numpy().reshape(-1) for _, labels in test_ds]).astype(int)
probabilities = model.predict(test_ds, verbose=0).reshape(-1)
y_pred = (probabilities >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"TEST ACCURACY: {acc:.4f} ({acc*100:.2f}%)")
print(f"PRECISION (open class): {precision:.4f}")
print(f"RECALL (open class):    {recall:.4f}")
print(f"F1 SCORE (open class):  {f1:.4f}")
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Held-out Test Set")
plt.show()


In [ ]:
# Save model and download it to your computer.
model.save("eye_cnn.keras")
from google.colab import files
files.download("eye_cnn.keras")


## Before using this model in the webcam app
1. Copy the downloaded `eye_cnn.keras` into `DriveSafe-AI/backend/models/`.
2. Restart Flask.
3. Check `http://127.0.0.1:5000/api/health`; it should show `model_loaded: true`.
4. Test with different people, glasses, camera distances and lighting. Report the observed limitations.

### Viva note
The sigmoid output is the estimated probability of the `open` class. In this notebook, `closed=0` and `open=1`. The backend classifies an eye as open when probability ≥ 0.5. This threshold is a simple starting point and should be validated on real test data.
